# Grocery Basket Optimizer — v1

This version converts the v0 proof of concept into reusable optimization logic.

The optimizer accepts:

- a grocery price dataset
- a user basket
- a maximum number of stores
- a minimum savings threshold

It returns:

- the best single-store option
- the best two-store option
- estimated savings
- an item-by-item shopping plan
- a practical recommendation

In [1]:
from itertools import combinations
from pathlib import Path

import pandas as pd

prices = pd.read_csv("../data/sample_prices.csv")
basket = pd.read_csv("../data/sample_basket.csv")

prices.head()

,item,category,store,price,unit,package_size,flyer_week
0,eggs,dairy,No Frills,3.99,dozen,12,2026-07-08
1,eggs,dairy,Walmart,4.29,dozen,12,2026-07-08
2,eggs,dairy,Food Basics,3.79,dozen,12,2026-07-08
3,milk,dairy,No Frills,5.89,4L,4,2026-07-08
4,milk,dairy,Walmart,5.69,4L,4,2026-07-08


In [2]:
print("Prices shape:", prices.shape)
print("Basket shape:", basket.shape)
print("Stores:", prices["store"].unique().tolist())

basket

Prices shape: (45, 7)
Basket shape: (10, 2)
Stores: ['No Frills', 'Walmart', 'Food Basics']


,item,quantity
0,eggs,1
1,milk,1
2,bread,2
3,chicken breast,1
4,pasta,2
5,apples,1
6,bananas,1
7,yogurt,1
8,cheese,1
9,frozen vegetables,2


In [3]:
def prepare_basket_data(
    prices: pd.DataFrame,
    basket: pd.DataFrame
) -> pd.DataFrame:
    """
    Validate the input tables and merge basket quantities with store prices.
    """

    required_price_columns = {"item", "store", "price"}
    required_basket_columns = {"item", "quantity"}

    missing_price_columns = required_price_columns - set(prices.columns)
    missing_basket_columns = required_basket_columns - set(basket.columns)

    if missing_price_columns:
        raise ValueError(
            f"Prices data is missing columns: {sorted(missing_price_columns)}"
        )

    if missing_basket_columns:
        raise ValueError(
            f"Basket data is missing columns: {sorted(missing_basket_columns)}"
        )

    if basket.empty:
        raise ValueError("The grocery basket cannot be empty.")

    if (basket["quantity"] <= 0).any():
        raise ValueError("Every basket quantity must be greater than zero.")

    prices_clean = prices.copy()
    basket_clean = basket.copy()

    # Standardize item names for more reliable matching
    prices_clean["item"] = prices_clean["item"].str.strip().str.lower()
    basket_clean["item"] = basket_clean["item"].str.strip().str.lower()

    # Combine duplicate basket items
    basket_clean = (
        basket_clean
        .groupby("item", as_index=False)["quantity"]
        .sum()
    )

    unavailable_items = sorted(
        set(basket_clean["item"]) - set(prices_clean["item"])
    )

    if unavailable_items:
        raise ValueError(
            f"No price data was found for: {unavailable_items}"
        )

    basket_prices = basket_clean.merge(
        prices_clean,
        on="item",
        how="left"
    )

    basket_prices["total_item_cost"] = (
        basket_prices["quantity"] * basket_prices["price"]
    )

    return basket_prices

In [4]:
def calculate_store_combo(
    basket_prices: pd.DataFrame,
    selected_stores: tuple[str, ...]
) -> dict:
    """
    Assign each basket item to its cheapest available store
    within the selected combination.
    """

    combo_data = basket_prices[
        basket_prices["store"].isin(selected_stores)
    ].copy()

    shopping_plan = (
        combo_data
        .sort_values(["item", "total_item_cost", "store"])
        .groupby("item", as_index=False)
        .first()
    )

    required_items = set(basket_prices["item"].unique())
    covered_items = set(shopping_plan["item"])

    if covered_items != required_items:
        return {
            "stores": selected_stores,
            "total_cost": float("inf"),
            "shopping_plan": None
        }

    return {
        "stores": selected_stores,
        "total_cost": shopping_plan["total_item_cost"].sum(),
        "shopping_plan": shopping_plan
    }

In [5]:
def optimize_basket(
    prices: pd.DataFrame,
    basket: pd.DataFrame,
    max_stores: int = 2,
    savings_threshold: float = 5.00
) -> dict:
    """
    Optimize a grocery basket for one or two stores.
    """

    if max_stores not in {1, 2}:
        raise ValueError("max_stores must currently be either 1 or 2.")

    if savings_threshold < 0:
        raise ValueError("savings_threshold cannot be negative.")

    basket_prices = prepare_basket_data(prices, basket)
    stores = sorted(basket_prices["store"].dropna().unique())

    # Test every single-store option
    single_store_results = [
        calculate_store_combo(basket_prices, (store,))
        for store in stores
    ]

    valid_single_results = [
        result
        for result in single_store_results
        if result["total_cost"] != float("inf")
    ]

    if not valid_single_results:
        raise ValueError(
            "No single store contains every item in the basket."
        )

    best_single = min(
        valid_single_results,
        key=lambda result: result["total_cost"]
    )

    # Users allowing only one store stop here
    if max_stores == 1:
        return {
            "best_single": best_single,
            "best_multi": None,
            "savings": 0.0,
            "worth_it": False,
            "recommended_option": best_single,
            "recommendation": (
                f"Shop at {best_single['stores'][0]}."
            )
        }

    # Test every two-store combination
    two_store_results = [
        calculate_store_combo(basket_prices, store_pair)
        for store_pair in combinations(stores, 2)
    ]

    valid_two_store_results = [
        result
        for result in two_store_results
        if result["total_cost"] != float("inf")
    ]

    if not valid_two_store_results:
        raise ValueError(
            "No valid two-store combination covers the full basket."
        )

    best_two = min(
        valid_two_store_results,
        key=lambda result: result["total_cost"]
    )

    savings = best_single["total_cost"] - best_two["total_cost"]
    worth_it = savings >= savings_threshold

    if worth_it:
        recommended_option = best_two
        recommendation = (
            "Visit two stores because the savings meet or exceed "
            "your minimum threshold."
        )
    else:
        recommended_option = best_single
        recommendation = (
            "Stick with one store because the extra savings do not "
            "meet your minimum threshold."
        )

    return {
        "best_single": best_single,
        "best_multi": best_two,
        "savings": savings,
        "worth_it": worth_it,
        "recommended_option": recommended_option,
        "recommendation": recommendation
    }

In [6]:
result = optimize_basket(
    prices=prices,
    basket=basket,
    max_stores=2,
    savings_threshold=5.00
)

In [7]:
best_single = result["best_single"]
best_two = result["best_multi"]

print("GROCERY BASKET OPTIMIZER SUMMARY")
print("--------------------------------")

print(
    f"Best single-store option: "
    f"{best_single['stores'][0]} — "
    f"${best_single['total_cost']:.2f}"
)

if best_two is not None:
    print(
        f"Best two-store option: "
        f"{' + '.join(best_two['stores'])} — "
        f"${best_two['total_cost']:.2f}"
    )

    print(f"Savings from two stores: ${result['savings']:.2f}")

print(f"Is the second store worth it? {'Yes' if result['worth_it'] else 'No'}")
print()
print(result["recommendation"])

GROCERY BASKET OPTIMIZER SUMMARY
--------------------------------
Best single-store option: Food Basics — $53.37
Best two-store option: Food Basics + Walmart — $51.07
Savings from two stores: $2.30
Is the second store worth it? No

Stick with one store because the extra savings do not meet your minimum threshold.


In [8]:
shopping_plan = result["recommended_option"]["shopping_plan"].copy()

shopping_plan = shopping_plan[
    ["item", "quantity", "store", "price", "total_item_cost"]
].sort_values(["store", "item"])

shopping_plan

,item,quantity,store,price,total_item_cost
0,apples,1,Food Basics,4.49,4.49
1,bananas,1,Food Basics,1.79,1.79
2,bread,2,Food Basics,3.19,6.38
3,cheese,1,Food Basics,4.99,4.99
4,chicken breast,1,Food Basics,11.99,11.99
5,eggs,1,Food Basics,3.79,3.79
6,frozen vegetables,2,Food Basics,3.29,6.58
7,milk,1,Food Basics,5.99,5.99
8,pasta,2,Food Basics,1.79,3.58
9,yogurt,1,Food Basics,3.79,3.79


In [9]:
if result["best_multi"] is not None:
    cheapest_two_store_plan = result["best_multi"]["shopping_plan"][
        ["item", "quantity", "store", "price", "total_item_cost"]
    ].sort_values(["store", "item"])

    display(cheapest_two_store_plan)

,item,quantity,store,price,total_item_cost
0,apples,1,Food Basics,4.49,4.49
3,cheese,1,Food Basics,4.99,4.99
4,chicken breast,1,Food Basics,11.99,11.99
5,eggs,1,Food Basics,3.79,3.79
8,pasta,2,Food Basics,1.79,3.58
9,yogurt,1,Food Basics,3.79,3.79
1,bananas,1,Walmart,1.59,1.59
2,bread,2,Walmart,2.79,5.58
6,frozen vegetables,2,Walmart,2.79,5.58
7,milk,1,Walmart,5.69,5.69


In [10]:
test_basket = pd.DataFrame({
    "item": [
        "milk",
        "bread",
        "chicken breast",
        "pasta",
        "cereal"
    ],
    "quantity": [1, 2, 1, 3, 1]
})

test_result = optimize_basket(
    prices=prices,
    basket=test_basket,
    max_stores=2,
    savings_threshold=3.00
)

print(test_result["recommendation"])
print(f"Savings: ${test_result['savings']:.2f}")

Stick with one store because the extra savings do not meet your minimum threshold.
Savings: $1.40


In [11]:
one_store_result = optimize_basket(
    prices=prices,
    basket=basket,
    max_stores=1,
    savings_threshold=5.00
)

print(one_store_result["recommendation"])

Shop at Food Basics.


In [12]:
invalid_basket = pd.DataFrame({
    "item": ["milk", "fake item"],
    "quantity": [1, 1]
})

try:
    optimize_basket(
        prices=prices,
        basket=invalid_basket,
        max_stores=2
    )
except ValueError as error:
    print(error)

No price data was found for: ['fake item']
